# Model 3: ordering plan — run notebook
Run top to bottom. Needs `model3.py` next to this notebook; finds `model1.py` and `model2.py` automatically.

1. **Setup**: paths, imports, team submission
2. **Load**: data, naive policy, capacity
3. **Backtest**: naive vs quick LP vs stochastic A / B on 9 past windows + an injected Kestria ban (scored like the competition; a few minutes)
4. **Tuning** (optional, slow): knob grid ranked by the worst window
5. **Fill the submission**: `order_units` + `share_*` in the team file (plan A), plus `submission_B.csv` (aggressive)
6. **Submission check**: the doc's pre-upload checklist, all columns
7. **Look at the plan**: shares and orders per material
8. **Phase 2**: re-plan from week 221 when the new data is in

## 1. Setup

In [0]:
import os
import sys
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)

d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, "Data")):
    d = os.path.dirname(d)
REPO_ROOT = d if os.path.isdir(os.path.join(d, "Data")) else "/Workspace/Repos/DSDG Hackathon/DSDG Hackathon"
DATA = os.path.join(REPO_ROOT, "Data")

def find_module(name, preferred):
    if os.path.exists(os.path.join(preferred, name)):
        return preferred
    found = [p for p in glob.glob(os.path.join(REPO_ROOT, "**", name), recursive=True) if "archive" not in p]
    if not found:
        raise FileNotFoundError(f"{name} not found under {REPO_ROOT}")
    return os.path.dirname(max(found, key=os.path.getmtime))

PIPE = os.path.join(REPO_ROOT, "Code Base", "pipelining")
MODEL1_DIR = find_module("model1.py", os.path.join(PIPE, "Model 1", "iter3"))
MODEL2_DIR = find_module("model2.py", os.path.join(PIPE, "Model 2", "iter3"))
for p in (MODEL1_DIR, MODEL2_DIR):
    if p not in sys.path:
        sys.path.insert(0, p)
import model1
import model2
import model3

TEAM_SUBMISSION = os.path.join(MODEL1_DIR, "outputs", "submission.csv")
OUT = os.path.join(os.getcwd(), "outputs")
os.makedirs(OUT, exist_ok=True)
print("DATA:           ", DATA)
print("model1.py from: ", MODEL1_DIR)
print("model2.py from: ", MODEL2_DIR)
print("team submission:", TEAM_SUBMISSION, "(exists)" if os.path.exists(TEAM_SUBMISSION) else "(not yet - run Model 1 and 2 first)")

## 2. Load data

In [0]:
od = model3.OrderData(model1.load_data(DATA))
P = od.P
PHASE2 = P.last_week > 208
print(f"supply data through week {P.last_week}, demand through week {od.last_demand_week}; Phase 2 loaded: {PHASE2}")

naive_src = {m: P.suppliers.country.values[u][np.argmax(od.normal[u])] for m, u in zip(P.materials, P.mat_units)}
print("\nnaive policy (from sample_submission): order = this week's demand, 100% from the largest supplier:")
print(naive_src)
cap = pd.DataFrame({"material": P.materials,
                    "mean_demand": [od.demand[i].mean() for i in range(len(P.materials))],
                    "total_capacity": [od.normal[u].sum() for u in P.mat_units],
                    "largest_supplier": [od.normal[u].max() for u in P.mat_units]})
cap["total/demand"] = cap.total_capacity / cap.mean_demand
cap["largest/demand"] = cap.largest_supplier / cap.mean_demand
cap.round(2)

## 3. Backtest
Each past 13-week window is planned from the week before it (Models 1 and 2 refit there, no later data), then played out against what really happened.
- **F**: (naive cost - our cost) / (naive - hindsight), clipped to [0, 1]
- **fill**: share of demand met in shock weeks + 4 recovery weeks
- **div**: 1 - HHI of shares, rescaled (single source 0, even split 1)
- **score** = 0.40 F + 0.21 fill + 0.09 div (Model 3's 70% of the total)

`183+kestria_ban` is window 183 with a Kestria export ban injected into the actuals from its 4th week (8 weeks, capacity 0, prices +25%): the kind of shock bigger than anything in training.

In [0]:
CFG_A = model3.Config()                 # tuned defaults (plan A)
CFG_B = model3.AGGRESSIVE               # doc's aggressive settings (plan B)
configs = {"quick": model3.replace(CFG_A, policy="quick"), "stochastic A": CFG_A, "stochastic B": CFG_B}

bt = model3.backtest(od, configs=configs, cfg=CFG_A)
print(bt.summary().round(3).to_string())
wins = bt.windows
order = sorted(wins.window.unique(), key=lambda s: (int(s.split("+")[0]), len(s)))
pivot = wins.pivot(index="window", columns="policy", values="score").reindex(order)
print("\nscore by window:")
print(pivot.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(16, 4.5))
pols = ["naive", "quick", "stochastic A", "stochastic B", "hindsight"]
colors = {"naive": "#bbbbbb", "quick": "#4C72B0", "stochastic A": "#DD8452", "stochastic B": "#C44E52", "hindsight": "#55A868"}
x = np.arange(len(pivot))
for i, p in enumerate(pols):
    ax[0].bar(x + (i - 2) * 0.16, pivot[p], 0.16, label=p, color=colors[p])
ax[0].set_xticks(x, pivot.index, rotation=30)
ax[0].set(title="Score by window (max 0.70)", ylabel="0.40 F + 0.21 fill + 0.09 div")
ax[0].legend(fontsize=8, ncol=5)
cv = wins.pivot(index="window", columns="policy", values="cost_vs_naive").reindex(order)
for i, p in enumerate(pols):
    ax[1].bar(x + (i - 2) * 0.16, 100 * cv[p], 0.16, label=p, color=colors[p])
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_xticks(x, cv.index, rotation=30)
ax[1].set(title="Total cost vs naive (%)", ylabel="%")
plt.tight_layout()
fig.savefig(os.path.join(OUT, "model3_backtest.png"), dpi=150)
plt.show()

## 4. Tuning (optional)
The doc's knob grid, ranked by the **worst** window (robustness beats the best average with a big shock coming). Slow: 37 settings x 10 windows. Skipped unless `RERUN_TUNING = True`. Plan A's defaults already come from this grid (the top settings were within seed noise of each other; omega 0.05 was the steadiest). If you re-tune and a different row clearly wins, set `CFG_A` to it.

In [0]:
RERUN_TUNING = False
tune_path = os.path.join(OUT, "tuning_model3.csv")
if RERUN_TUNING:
    grid = {"omega": [0.0, 0.05, 0.15], "s_min": [0.1, 0.2, 0.33], "lam": [5.0, 8.0], "prob_mult": [1.0, 1.5]}
    table, _ = model3.tune(od, grid, base=model3.replace(CFG_A, n_scenarios=200))
    table.to_csv(tune_path)
    display_table = table.round(3).head(15)
elif os.path.exists(tune_path):
    display_table = pd.read_csv(tune_path, index_col=0).round(3).head(15)
else:
    display_table = None
    print("Tuning skipped (set RERUN_TUNING = True; 37 settings x 10 windows takes 10+ minutes).")
    print("Plan A's defaults came from this grid: omega 0.05, s_min 0.1, lambda 5, prob_mult 1.0.")
display_table

## 5. Fill the submission
Plan A (tuned) goes into the team file: `order_units` and the six `share_*` columns; `disruption_prob` and `price_forecast` stay as Models 1 and 2 wrote them. Plan B (aggressive: more stress weight, higher probabilities, lambda 8, even split) is written to `outputs/submission_B.csv` as the second final pick. Kaggle keeps the better of the two.

In [0]:
sub_in = TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else os.path.join(DATA, "sample_submission.csv")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    sub_A = model3.fill_submission(od, sub_in, CFG_A)
    sub_B = model3.fill_submission(od, sub_in, CFG_B)
if os.path.exists(TEAM_SUBMISSION):
    sub_A.to_csv(TEAM_SUBMISSION, index=False)
sub_A.to_csv(os.path.join(OUT, "submission_A.csv"), index=False)
sub_B.to_csv(os.path.join(OUT, "submission_B.csv"), index=False)
print("read: ", sub_in)
print("wrote plan A to", TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else "", "and", os.path.join(OUT, "submission_A.csv"))
print("wrote plan B to", os.path.join(OUT, "submission_B.csv"))
if not PHASE2:
    print("note: window 222-234 is planned from week 208 until Phase 2 data is loaded")
sub_A.head(12)

## 6. Submission check (all columns)

In [0]:
def check_submission(sub, suppliers):
    issues = []
    if len(sub) != 260:
        issues.append(f"{len(sub)} rows, expected 260")
    if sub.id.duplicated().any():
        issues.append("duplicate ids")
    if sub.isna().any().any():
        issues.append(f"missing values in {sub.columns[sub.isna().any()].tolist()}")
    if not sub.disruption_prob.between(0, 1).all():
        issues.append("disruption_prob outside [0, 1]")
    if not (sub.price_forecast > 0).all():
        issues.append("price_forecast <= 0")
    if (sub.order_units < 0).any() or not np.allclose(sub.order_units, sub.order_units.round()):
        issues.append("order_units negative or not whole numbers")
    share_cols = [c for c in sub.columns if c.startswith("share_")]
    if not np.allclose(sub[share_cols].sum(axis=1), 1.0, atol=1e-6):
        issues.append("shares don't sum to 1 on every row")
    mat = sub.id.str.extract(r"^w\d+_(.+)$")[0]
    sells = suppliers.groupby("material").country.apply(set)
    for col in share_cols:
        cty = col.split("_", 1)[1]
        bad = (sub[col] > 0) & ~mat.map(lambda m: cty in sells[m])
        if bad.any():
            issues.append(f"{col} > 0 for materials {cty} doesn't sell: {sorted(mat[bad].unique())}")
    return issues

sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))
for name, s in (("A", sub_A), ("B", sub_B)):
    problems = check_submission(s, P.suppliers)
    print(f"plan {name}:", "READY TO UPLOAD" if not problems else "FIX BEFORE UPLOAD:\n- " + "\n- ".join(problems))
    for col, who in (("disruption_prob", "Model 1"), ("price_forecast", "Model 2")):
        if (s[col] == sample[col]).all():
            print(f"  note: {col} is still the sample's - run {who}'s notebook first")

## 7. Look at the plan (window 209-221, plan A)

In [0]:
wk = sub_A.id.str.extract(r"^w(\d+)_(.+)$")
plan = sub_A.assign(week=wk[0].astype(int), material=wk[1])
w1 = plan[plan.week.between(209, 221)]
share_cols = [c for c in plan.columns if c.startswith("share_")]
mean_shares = w1.groupby("material")[share_cols].mean()
units = w1.groupby("material").order_units.sum()
dem = pd.Series({m: od.d(i, 209, 13).sum() for i, m in enumerate(P.materials)})
start = pd.Series({m: od.start_stock(i, 209, CFG_A) for i, m in enumerate(P.materials)})
overview = mean_shares.round(2)
overview["ordered_units"] = units
overview["ordered/demand"] = (units / dem).round(2)
overview["start_stock/demand"] = (start / dem).round(2)
print(overview.to_string())

fig, ax = plt.subplots(figsize=(11, 4.5))
bottom = np.zeros(len(mean_shares))
palette = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
for col, c in zip(share_cols, palette):
    ax.bar(mean_shares.index, mean_shares[col], bottom=bottom, label=col.replace("share_", ""), color=c)
    bottom += mean_shares[col].values
ax.set(title="Plan A: average share per supplier country, weeks 209-221", ylabel="share of orders")
ax.legend(ncol=6, fontsize=8)
plt.xticks(rotation=30)
plt.tight_layout()
fig.savefig(os.path.join(OUT, "model3_shares.png"), dpi=150)
plt.show()

## 8. Phase 2
Does nothing until `Data/` has weeks after 208. Then: re-run Models 1 and 2 first (their notebooks), then this cell re-plans window 222-234 from week 221 (A and B) and re-scores the backtest including window 209.

In [0]:
od = model3.OrderData(model1.load_data(DATA))
P = od.P
if P.last_week <= 208:
    print(f"No Phase 2 data yet (supply data ends at week {P.last_week}). Nothing to do.")
else:
    sub_in = TEAM_SUBMISSION if os.path.exists(TEAM_SUBMISSION) else os.path.join(DATA, "sample_submission.csv")
    sub_A = model3.fill_submission(od, sub_in, CFG_A)
    sub_B = model3.fill_submission(od, sub_in, CFG_B)
    if os.path.exists(TEAM_SUBMISSION):
        sub_A.to_csv(TEAM_SUBMISSION, index=False)
    sub_A.to_csv(os.path.join(OUT, "submission_A.csv"), index=False)
    sub_B.to_csv(os.path.join(OUT, "submission_B.csv"), index=False)
    for name, s in (("A", sub_A), ("B", sub_B)):
        problems = check_submission(s, P.suppliers)
        print(f"plan {name}:", "READY TO UPLOAD" if not problems else problems)
    windows = tuple(w for w in model3.DEFAULT_WINDOWS + (209,) if w + 12 <= P.last_week)
    print(model3.backtest(od, windows=windows, configs={"stochastic A": CFG_A, "stochastic B": CFG_B}, cfg=CFG_A).summary().round(3))